In [1]:
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.docstore.document import Document
import os
from dotenv import load_dotenv
import gradio as gr
from langchain.chains import RetrievalQA
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import RateLimitError
import time

# Load environment variables from .env file
load_dotenv()

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Validate API key
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key not found. Please set OPENAI_API_KEY in the script.")

def scrape_url(url):
    """Scrape a single URL and return the title and content."""
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            print(f"Non-200 status code for {url}: {r.status_code}")
            return None
        page = BeautifulSoup(r.text, "lxml")
        
        title = page.find("title").get_text(strip=True) if page.find("title") else ""
        main = page.select_one("article") or page.select_one("main") or page.body
        if not main:
            print(f"No main content found for {url}")
            return None
        
        for tag in main.find_all(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()
        
        text = re.sub(r"\s+", " ", main.get_text(" ", strip=True))
        return {"url": url, "title": title, "content": text}
    except Exception as e:
        print(f"Failed on {url}: {e}")
        return None

def scrape_urls(urls, output_file):
    """Scrape a list of URLs and save the data to a JSON file."""
    data = []
    for url in tqdm(urls):
        result = scrape_url(url)
        if result and result["content"]:
            data.append(result)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Saved {len(data)} pages to {output_file}")

# List of specific URLs to scrape
urls = [
    "https://meinstudium.fau.eu/studiengang/electromobility-aces-msc/",
    "https://meinstudium.fau.eu/studiengang/information-and-communication-technology-iuk-msc/",
    "https://meinstudium.fau.eu/studiengang/mathematics-msc/",
    "https://meinstudium.fau.eu/studiengang/medical-engineering-msc/",
    "https://meinstudium.fau.eu/studiengang/physics-msc/",
    "https://meinstudium.fau.eu/studiengang/psychology-msc/",
    "https://meinstudium.fau.eu/studiengang/sustainability-management-mba/",
    "https://meinstudium.fau.eu/studiengang/political-science-ma/",
    "https://meinstudium.fau.eu/studiengang/philosophy-ma/",
    "https://meinstudium.fau.eu/studiengang/energy-technology-msc/"
]

scrape_urls(urls, "fau_pages.json")

def load_data(json_file):
    """Load data from a JSON file."""
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: {json_file} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to parse {json_file}.")
        return []

def create_documents(data):
    """Create Document objects from the loaded data."""
    docs = []
    for entry in data:
        if all(key in entry for key in ["url", "title", "content"]):
            docs.append(Document(
                page_content=entry["content"],
                metadata={"title": entry["title"], "url": entry["url"]}
            ))
        else:
            print(f"Skipping invalid entry: {entry.get('url', 'unknown')}")
    return docs

def split_documents(docs):
    """Split the documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    return splitter.split_documents(docs)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=4, max=60),
    retry=retry_if_exception_type(RateLimitError)
)
def create_vectorstore(documents, persist_directory):
    """Create a Chroma vector store from the documents in batches."""
    try:
        embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key=OPENAI_API_KEY)
    except Exception as e:
        print(f"Error initializing embeddings: {e}")
        return None
    
    try:
        # Check if vector store already exists
        if os.path.exists(persist_directory):
            print(f"Vector store already exists at {persist_directory}. Loading existing store.")
            return Chroma(persist_directory=persist_directory, embedding_function=embeddings)
        
        # Process documents in batches to avoid rate limits
        batch_size = 10  # Adjust based on your rate limit
        vectorstore = None
        for i in tqdm(range(0, len(documents), batch_size), desc="Embedding documents"):
            batch = documents[i:i + batch_size]
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=batch,
                    embedding=embeddings,
                    persist_directory=persist_directory
                )
            else:
                vectorstore.add_documents(documents=batch)
            time.sleep(1)  # Small delay to respect rate limits
        
        vectorstore.persist()
        print(f"✅ Index built and stored in {persist_directory}")
        return vectorstore
    except Exception as e:
        print(f"Error building or persisting vector store: {e}")
        return None

def main():
    json_file = "fau_pages.json"
    data = load_data(json_file)
    print(f"Loaded {len(data)} entries from {json_file}")
    
    docs = create_documents(data)
    print(f"Created {len(docs)} documents")
    
    documents = split_documents(docs)
    print(f"Created {len(documents)} chunks")
    
    persist_directory = "./chroma_fau_openai"
    
    vectorstore = create_vectorstore(documents, persist_directory)
    if vectorstore is None:
        print("Failed to create vector store.")
        return
    
    llm = ChatOpenAI(model="gpt-3.5-turbo", openai_api_key=OPENAI_API_KEY, max_retries=5)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True
    )
    
    query = "I want to apply to mba courses"
    response = qa.invoke(query)
    print("Answer:\n", response["result"])
    print("\nSources:")
    for doc in response["source_documents"]:
        print("-", doc.metadata["url"])

# Load the vector store
persist_directory = "./chroma_fau_openai"
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key=OPENAI_API_KEY)
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

# Create the LLM and RetrievalQA chain
llm = ChatOpenAI(model="gpt-3.5-turbo", openai_api_key=OPENAI_API_KEY, max_retries=5)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

def answer_query(query):
    response = qa.invoke(query)
    answer = response["result"]
    sources = "\n".join([doc.metadata["url"] for doc in response["source_documents"]])
    return answer, sources

# Create the Gradio interface
demo = gr.Blocks()
with demo:
    gr.Markdown("# FAU Erlangen Master's Programs Query Interface")
    gr.Markdown("Ask a question about master's programs at FAU Erlangen, and I'll do my best to provide an answer based on the available information.")
    
    gr.Markdown("### Available Courses")
    with gr.Row():
        gr.Markdown("- Electromobility (M.Sc.)")
        gr.Markdown("- Info & Comm Tech (M.Sc.)")
        gr.Markdown("- Mathematics (M.Sc.)")
        gr.Markdown("- Medical Engineering (M.Sc.)")
        gr.Markdown("- Physics (M.Sc.)")
    with gr.Row():
        gr.Markdown("- Psychology (M.Sc.)")
        gr.Markdown("- Sustainability Mgmt (MBA)")
        gr.Markdown("- Political Science (M.A.)")
        gr.Markdown("- Philosophy (M.A.)")
        gr.Markdown("- Energy Technology (M.Sc.)")
    
    gr.Markdown("### Example Queries")
    gr.Markdown("""
    - Tell me about the Mathematics M.Sc. program.
    - How can I apply to the Sustainability Management MBA?
    - What is the curriculum for Information and Communication Technology (IUK) M.Sc.?
    """)
    
    with gr.Row():
        query = gr.Textbox(label="Your Query")
        submit = gr.Button("Submit")
    
    with gr.Row():
        answer = gr.Textbox(label="Answer", lines=10)
        sources = gr.Textbox(label="Sources", lines=5)
    
    submit.click(
        answer_query,
        inputs=query,
        outputs=[answer, sources]
    )

if __name__ == "__main__":
    main()
    demo.launch()

ModuleNotFoundError: No module named 'langchain_openai'

In [2]:
!pip install requests beautifulsoup4 lxml tqdm langchain langchain-community chromadb openai gradio tenacity python-dotenv       

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.7 kB ? eta -:--:--
     --------------------------------- ---- 51.2/57.7 kB 525.1 kB/s eta 0:00:01
     -------------------------------------- 57.7/57.7 kB 504.6 kB/s eta 0:00:00
  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_core-0.3.76-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
     ---------------------------------------- 0.0/68.4 kB ? eta -:--:--
     ----------------------------------- ---- 61.4/68.4 kB 1.7 MB/s eta 0:00:01
     ---------------------------------------- 68.4/68.4 kB 1.2 MB/s eta 0:00:00
  Using cached dataclasses_


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\MSI\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
